In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 17:36:30--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.222, 18.155.128.187, 18.155.128.6, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.222|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   110MB/s    in 0.6s    

2026-03-08 17:36:31 (110 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [4]:
!wc -l yellow_tripdata_2025-11.parquet

282291 yellow_tripdata_2025-11.parquet


In [3]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [6]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [13]:
df = df.repartition(4)

In [8]:
df.write.parquet('yellow/2025/11/')

In [5]:
df.createOrReplaceTempView('trips_data')

In [17]:
spark.sql("""
    SELECT
        COUNT(*)
    FROM
        trips_data
    WHERE (to_date(tpep_pickup_datetime) = DATE('2025-11-15'))

""").show()

+--------------------+
|count(trip_distance)|
+--------------------+
|              162604|
+--------------------+



In [18]:
spark.sql("""
    SELECT
        MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime))/3600) as max_duration
    FROM trips_data
""").show()

+-----------------+
|     max_duration|
+-----------------+
|90.64666666666666|
+-----------------+



In [19]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 22:01:35--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.6, 18.155.128.187, 18.155.128.222, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.6|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.2’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-08 22:01:35 (188 MB/s) - ‘taxi_zone_lookup.csv.2’ saved [12331/12331]



In [22]:
df_taxi_zone_lookup = spark \
                    .read \
                    .option("header", "true") \
                    .csv("taxi_zone_lookup.csv")

In [26]:
df_taxi_zone_lookup.schema


StructType([StructField('LocationID', StringType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [28]:
from pyspark.sql import types

In [29]:
schema =  types.StructType([
    types.StructField('LocationID', types.IntegerType(), True),
    types.StructField('Borough', types.StringType(), True),
    types.StructField('Zone', types.StringType(), True),
    types.StructField('service_zone', types.StringType(), True)
])

In [33]:
df_taxi_zone_lookup = spark \
                    .read \
                    .option("header", "true") \
                    .schema(schema) \
                    .csv("taxi_zone_lookup.csv")

In [34]:
df_taxi_zone_lookup.show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


In [36]:
df_taxi_zone_lookup.createOrReplaceTempView('taxi_zone_lookup')

In [42]:
spark.sql("""
WITH count_pickup_location AS (
    SELECT
        PULocationID,
        COUNT(*) as pickup_number
    FROM trips_data
    GROUP BY PULocationID
    )
SELECT 
    t.Borough,
    t.Zone,
    c.pickup_number
FROM count_pickup_location c
LEFT JOIN taxi_zone_lookup t
ON c.PULocationID = t.LocationID
ORDER BY c.pickup_number
LIMIT 10
""").show()

+-------------+--------------------+-------------+
|      Borough|                Zone|pickup_number|
+-------------+--------------------+-------------+
|    Manhattan|Governor's Island...|            1|
|Staten Island|       Arden Heights|            1|
|Staten Island|Eltingville/Annad...|            1|
|Staten Island|       Port Richmond|            3|
|Staten Island|   Rossville/Woodrow|            4|
|        Bronx|       Rikers Island|            4|
|     Brooklyn| Green-Wood Cemetery|            4|
|Staten Island|         Great Kills|            4|
|       Queens|         Jamaica Bay|            5|
|Staten Island|         Westerleigh|           12|
+-------------+--------------------+-------------+



In [35]:
spark.stop()